# Promoted mistake pipeline — saved audit review

Run All here **only reads saved results**; it does not run pitch inference, alignment, or modify production. Use `mistake.ipynb` to recompute the benchmark. Current production uses repeat-only refinement, a 100 ms / 0.5 note-length rule, 0.75 pitch step, 40 ms silence, and pitch/time alignment weights 4/1.

Start with the remaining false positives and missed events. Classifications are diagnostic clues, not proven causes; inspect the case snapshot before changing algorithms. This is the tuning corpus, not held-out validation.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
import sys

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from benchmarks.modules.mistake.MistakeNotebook import MistakeNotebook
from benchmarks.modules.mistake.provenance.PipelineAudit import summary

OUTPUT = REPO_ROOT / "benchmarks/results/mistake_competitors_coco_promoted"
AUDIT = OUTPUT / "pipeline_audit"
SYMBOLIC = REPO_ROOT / "benchmarks/results/mistake_competitors_coco_symbolic_promoted"
METHOD = "Attune (repeat only)"
run = json.loads((OUTPUT / "run.json").read_text())
assert run["status"] == "complete"
rows = pd.read_csv(OUTPUT / "rows.csv")
symbolic = pd.read_csv(SYMBOLIC / "rows.csv")
report = MistakeNotebook.overview(OUTPUT, symbolic_rows=symbolic)
display(report["leaderboard"])
display(report["clean"])
display(report["extraction"])


## Reproduced stage metrics

Pitch-frame, note-event, and mistake-event F1 measure different things. All saved audited method/case counts must reproduce the benchmark.


In [ ]:
reproduction = pd.read_csv(AUDIT / "reproduction.csv")
assert reproduction.reproduced.all()
print(f"Reproduced {len(reproduction)} saved case/method results")
display(summary(OUTPUT))


## Remaining mistake events

Sort by the total number of incorrect events. Clean controls are excluded here and shown in the table above.


In [ ]:
events = pd.read_csv(AUDIT / "events.csv")
remaining = events[
    events.method.eq(METHOD) & events.rate.gt(0) & ~events.correct
].copy()
remaining["error"] = remaining.truth.map({True: "FN", False: "FP"})
display(
    remaining.groupby(["instrument", "seed", "error", "kind"])
    .size()
    .unstack(["error", "kind"], fill_value=0)
)
ranked = (
    remaining.groupby(["case_id", "instrument", "seed"])
    .size()
    .rename("incorrect_events")
    .sort_values(ascending=False)
)
display(ranked.to_frame())
causes = pd.read_csv(AUDIT / "false_positive_causes.csv")
display(
    causes[causes.method.eq(METHOD)]
    .groupby(["kind", "reason"])
    .size()
    .rename("FP")
    .to_frame()
)


## Inspect one case

Change `CASE_ID` to any ID above. The snapshot contains fitted score notes, detected notes, alignment pairs, predictions and truth. Pair entries index the snapshot’s note/score arrays. Compare the performed MIDI with detected boundaries before blaming the cost model.


In [ ]:
CASE_ID = ranked.index[0][0] if len(ranked) else rows.case_id.iloc[0]
snapshot_path = AUDIT / f"{CASE_ID}.json"
snapshot = json.loads(snapshot_path.read_text())
print("Snapshot:", snapshot_path)
display(remaining[remaining.case_id.eq(CASE_ID)])
display(pd.DataFrame(snapshot["performed"]))
display(pd.DataFrame(snapshot[METHOD]["notes"]))
display(
    pd.DataFrame(
        snapshot[METHOD]["pairs"], columns=["detected_note_index", "score_note_index"]
    )
)
display(pd.DataFrame(snapshot[METHOD]["score"]))


## Extraction misses and exploratory probes

Oracle pitch uses idealized MIDI occupancy, not an achievable acoustic detector. Pitch weight 4 is now the baseline; that ablation is a replay control, not a new improvement. Keep new ideas separate from promotion until they improve held-out results and clean controls.


In [ ]:
missed = pd.read_csv(AUDIT / "missed_notes.csv")
display(
    missed[missed.tolerance.eq(0.1)].sort_values(
        ["support", "duration"], ascending=[False, True]
    )
)
ablations = pd.read_csv(AUDIT / "alignment_ablations.csv")
counts = ablations.groupby("method")[["tp", "fp", "fn"]].sum()
counts["F1 %"] = 200 * counts.tp / (2 * counts.tp + counts.fp + counts.fn)
display(counts.round(3))
print("Next-session checklist: docs/mistake-benchmark-promoted-2026-09-27.md")
